<a href="https://colab.research.google.com/github/skeieowk/linear/blob/main/pandas_12%EC%A3%BC%EC%B0%A8_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 파이썬 빅데이터시각화 12주차 – Pandas 활용
피벗 테이블 · 피벗 테이블의 활용 · 데이터 집계(groupby)

위에서부터 순서대로 실행하세요. (런타임 → 모두 실행도 가능)

## 준비: 드라이브 연결 및 데이터 파일 확인
`sales.xlsx`, `crime_in_seoul.csv`, `gas_stations.csv`를 **내 드라이브(MyDrive)** 에 업로드해 두세요.
(`fruits.xlsx`는 없으면 슬라이드의 표와 같은 데이터로 자동 생성됩니다.)

In [8]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive


In [9]:
import os
import pandas as pd
from google.colab import files

# fruits.xlsx가 없으면 슬라이드(3강 풀이)와 같은 데이터로 생성
if not os.path.exists('fruits.xlsx'):
    pd.DataFrame({
        'city':     ['부산','부산','부산','부산','서울','서울','서울'],
        'fruits':   ['apple','orange','banana','banana','apple','apple','banana'],
        'price':    [100, 200, 250, 300, 150, 200, 400],
        'quantity': [1, 2, 3, 4, 5, 6, 7],
    }).to_excel('fruits.xlsx', index=False)
    print("fruits.xlsx 생성 완료")

need = ['sales.xlsx', 'crime_in_seoul.csv', 'gas_stations.csv']
missing = [f for f in need if not os.path.exists(f)]
if missing:
    print("현재 폴더에 없는 파일:", missing)
    print("아래 업로드 창에서 선택하세요. (현재 폴더에 저장됩니다)")
    uploaded = files.upload()
else:
    print("필요한 파일이 모두 있습니다.")

현재 폴더에 없는 파일: ['sales.xlsx']
아래 업로드 창에서 선택하세요. (현재 폴더에 저장됩니다)


Saving crime_in_seoul.csv to crime_in_seoul (1).csv
Saving fruits.xlsx to fruits (1).xlsx
Saving gas_stations.csv to gas_stations (1).csv
Saving sales.xlsx to sales.xlsx


## 1강. 피벗 테이블을 이용한 데이터 정리

### 1. 피벗 테이블
대화형 테이블의 일종으로, 데이터의 나열 형태에 따라서 집계나 카운트 등의 계산을 하는 기능

`pd.pivot_table(DataFrame객체, index=인덱스 리스트, values=값 리스트)`
- index: 인덱스로 사용할 컬럼 리스트
- values: 집계 가능한 값을 가지는 컬럼 리스트

### 2. 데이터 읽기

In [10]:
import pandas as pd

sales = pd.read_excel('sales.xlsx')
sales.head()

,Account,Name,Rep,Manager,Product,Quantity,Price,Status
0,714466,Trantow-Barrows,Craig Booker,Debra Henley,CPU,1,30000,presented
1,714466,Trantow-Barrows,Craig Booker,Debra Henley,Software,1,10000,presented
2,714466,Trantow-Barrows,Craig Booker,Debra Henley,Maintenance,2,5000,pending
3,737550,"Fritsch, Russel and Anderson",Craig Booker,Debra Henley,CPU,1,35000,declined
4,146832,Kiehn-Spinka,Daniel Hilton,Debra Henley,CPU,2,65000,won


### 3. 피벗 테이블 만들기

In [11]:
pd.pivot_table(sales, index=['Name'], values=['Price', 'Quantity'])

,Price,Quantity
Name,,
Barton LLC,35000.0,1.000000
"Fritsch, Russel and Anderson",35000.0,1.000000
Herman LLC,65000.0,2.000000
Jerde-Hilpert,5000.0,2.000000
"Kassulke, Ondricka and Metz",7000.0,3.000000
Keeling LLC,100000.0,5.000000
Kiehn-Spinka,65000.0,2.000000
Koepp Ltd,35000.0,2.000000
Kulas Inc,25000.0,1.500000


In [12]:
pd.pivot_table(sales, index=['Name', 'Rep', 'Manager'], values=['Price', 'Quantity'])

,,,Price,Quantity
Name,Rep,Manager,,
Barton LLC,John Smith,Debra Henley,35000.0,1.000000
"Fritsch, Russel and Anderson",Craig Booker,Debra Henley,35000.0,1.000000
Herman LLC,Cedric Moss,Fred Anderson,65000.0,2.000000
Jerde-Hilpert,John Smith,Debra Henley,5000.0,2.000000
"Kassulke, Ondricka and Metz",Wendy Yule,Fred Anderson,7000.0,3.000000
Keeling LLC,Wendy Yule,Fred Anderson,100000.0,5.000000
Kiehn-Spinka,Daniel Hilton,Debra Henley,65000.0,2.000000
Koepp Ltd,Wendy Yule,Fred Anderson,35000.0,2.000000
Kulas Inc,Daniel Hilton,Debra Henley,25000.0,1.500000


In [13]:
# value를 pivot_table로 합친 경우 평균치가 기본이 됨
pd.pivot_table(sales, index=['Manager', 'Rep'], values=['Price'])

Price
Manager       Rep                        
Debra Henley  Craig Booker   20000.000000
              Daniel Hilton  38333.333333
              John Smith     20000.000000
Fred Anderson Cedric Moss    27500.000000
              Wendy Yule     44250.000000

In [14]:
# value의 값을 합산으로 하고 싶을 때 (슬라이드의 aggfunc=sum과 같은 결과)
pd.pivot_table(sales, index=['Manager', 'Rep'], values=['Price'], aggfunc='sum')

Price
Manager       Rep                  
Debra Henley  Craig Booker    80000
              Daniel Hilton  115000
              John Smith      40000
Fred Anderson Cedric Moss    110000
              Wendy Yule     177000

In [15]:
pd.pivot_table(sales, index=['Manager', 'Rep', 'Product'], values=['Price', 'Quantity'], aggfunc='sum')

Price  Quantity
Manager       Rep           Product                      
Debra Henley  Craig Booker  CPU           65000         2
                            Maintenance    5000         2
                            Software      10000         1
              Daniel Hilton CPU          105000         4
                            Software      10000         1
              John Smith    CPU           35000         1
                            Maintenance    5000         2
Fred Anderson Cedric Moss   CPU           95000         3
                            Maintenance    5000         1
                            Software      10000         1
              Wendy Yule    CPU          165000         7
                            Maintenance    7000         3
                            Monitor        5000         2

## 2강. 피벗 테이블의 활용

### 문제 1·2. crime_in_seoul.csv

csv 파일을 읽어서 DataFrame 만들기 (인코딩은 'euc-kr')

In [16]:
crime_seoul = pd.read_csv('crime_in_seoul.csv', encoding='euc-kr')
crime_seoul

,구분,죄종,발생검거,건수
0,중부,살인,발생,1
1,중부,살인,검거,1
2,중부,강도,발생,5
3,중부,강도,검거,7
4,중부,강간,발생,132
...,...,...,...,...
305,수서,강간,검거,171
306,수서,절도,발생,984
307,수서,절도,검거,481
308,수서,폭력,발생,1697


살인의 발생, 검거에 대한 합계 구하기 (피벗 테이블 사용)

In [17]:
x = (crime_seoul.죄종 == '살인')
d = crime_seoul[x]
pd.pivot_table(d, index=['발생검거'], values=['건수'], aggfunc='sum')

,건수
발생검거,
검거,137
발생,143


죄종에 따른 발생, 검거 건수 합계 구하기 (피벗 테이블 사용)

In [18]:
pd.pivot_table(crime_seoul, index=['죄종', '발생검거'], values=['건수'], aggfunc='sum')

건수
죄종 발생검거       
강간 검거     5118
   발생     6412
강도 검거      146
   발생      143
살인 검거      137
   발생      143
절도 검거    19531
   발생    39175
폭력 검거    48264
   발생    56075

### 문제 3·4. gas_stations.csv

csv 파일을 읽어서 DataFrame 만들기 (인코딩은 'euc-kr')

In [19]:
d = pd.read_csv('gas_stations.csv', encoding='euc-kr')
d

,Unnamed: 0,상호,주소,가격,셀프,상표,구
0,0,구천면주유소,서울 강동구 구천면로 357 (암사동),1585.0,N,현대오일뱅크,강동구
1,1,지에스칼텍스㈜ 동서울주유소,서울 강동구 천호대로 1456 (상일동),1595.0,Y,GS칼텍스,강동구
2,2,주)지유에너지직영 오렌지주유소,서울 강동구 성안로 102 (성내동),1596.0,N,SK에너지,강동구
3,3,천호현대주유소,서울 강동구 천중로 67 (천호동),1596.0,N,현대오일뱅크,강동구
4,4,지에스칼텍스㈜ 신월주유소,서울 강동구 양재대로 1323 (성내동),1598.0,N,GS칼텍스,강동구
...,...,...,...,...,...,...,...
487,487,(주)소모에너지엔테크놀러지 쎈트럴주유소,서울 강남구 삼성로 335 (대치동),1998.0,N,GS칼텍스,강남구
488,488,(주)만정에너지,서울 강남구 봉은사로 433 (삼성동),1999.0,N,GS칼텍스,강남구
489,489,금성주유소,서울 강남구 테헤란로 619 (삼성동),1999.0,N,SK에너지,강남구
490,490,오천주유소,서울 강남구 봉은사로 503 (삼성동),1999.0,N,SK에너지,강남구


구별 평균 주유 가격 추출하기 (피벗 테이블 사용)

In [20]:
pd.pivot_table(d, index=['구'], values=['가격'])

,가격
구,
강남구,1772.900000
강동구,1648.250000
강북구,1542.846154
강서구,1593.029412
관악구,1612.000000
광진구,1574.235294
구로구,1618.523810
금천구,1579.153846
노원구,1621.466667


평균 주유 가격이 저렴한 순서대로 7개 구 추출하기 (피벗 테이블 사용)

In [21]:
gu_data = pd.pivot_table(d, index=['구'], values=['가격'])
sorted_gu = gu_data.sort_values(by='가격', ascending=True)
sorted_gu.head(7)

,가격
구,
강북구,1542.846154
중랑구,1546.625000
은평구,1559.312500
성북구,1559.652174
도봉구,1564.222222
광진구,1574.235294
금천구,1579.153846


## 3강. 데이터 집계의 이해와 활용

### 1. 그룹화
`DataFrame객체.groupby(기준이 되는 열 또는 열의 리스트)`

In [22]:
names = ['철수', '영이', '길동', '미영', '순이', '철이']
columns = ['성별', '점수']
data = [['남',90],['여',90],['남',85],['여',99],['여',76],['남',98]]

d1 = pd.DataFrame(data, index=names, columns=columns)
d1

,성별,점수
철수,남,90
영이,여,90
길동,남,85
미영,여,99
순이,여,76
철이,남,98


In [23]:
grouped = d1.groupby('성별')

for key, group in grouped:
    print('* key:', key)
    print('* number:', len(group))
    print(group)
    print()

* key: 남
* number: 3
   성별  점수
철수  남  90
길동  남  85
철이  남  98

* key: 여
* number: 3
   성별  점수
영이  여  90
미영  여  99
순이  여  76



### 2. 그룹 연산
`group객체.함수()` — 함수: mean(), max(), min(), count(), std() 등

In [24]:
grouped['점수'].max()

,점수
성별,
남,98
여,99


In [25]:
grouped['점수'].min()

,점수
성별,
남,85
여,76


In [26]:
grouped['점수'].mean()

,점수
성별,
남,91.000000
여,88.333333


In [27]:
grouped['점수'].std()

,점수
성별,
남,6.557439
여,11.590226


### 3·4. 문제와 풀이 – fruits.xlsx

엑셀 파일을 읽어서 DataFrame 만들기

In [28]:
fruits = pd.read_excel('fruits.xlsx')
fruits

,city,fruits,price,quantity
0,부산,apple,100,1
1,부산,orange,200,2
2,부산,banana,250,3
3,부산,banana,300,4
4,서울,apple,150,5
5,서울,apple,200,6
6,서울,banana,400,7


도시 별 가격과 수량의 평균값 구하기 (그룹화하여 집계)

In [29]:
grouped = fruits.groupby('city')
grouped[['price','quantity']].mean()

,price,quantity
city,,
부산,212.5,2.5
서울,250.0,6.0


도시의 과일 별 가격과 수량의 평균값 구하기 (그룹화하여 집계)

In [30]:
grouped = fruits.groupby(['city', 'fruits'])
grouped[['price','quantity']].mean()

price  quantity
city fruits                 
부산   apple   100.0       1.0
     banana  275.0       3.5
     orange  200.0       2.0
서울   apple   175.0       5.5
     banana  400.0       7.0

도시의 과일 별 가격의 최소값과 최대값 구하기 (그룹화하여 집계)

In [31]:
grouped = fruits.groupby(['city', 'fruits'])
grouped[['price']].agg(['min','max'])

price     
              min  max
city fruits           
부산   apple    100  100
     banana   250  300
     orange   200  200
서울   apple    150  200
     banana   400  400